## **1.0 Prompt Engineering for Item Generation**



### **Overview**

This hands-on workshop covers various prompt engineering techniques for generating high-quality items.

Learning Objectives:
- Master different types of prompting strategies
- Implement structured prompt templates for item generation
- Apply few-shot learning for consistent item quality
- Use chain-of-thought reasoning for complex assessments
- Implement evaluation and refinement workflows

### **Basic Concept**
Prompt engineering is the process of crafting effective instructions for language models to get the best possible results. It means carefully choosing words, providing context, and structuring requests in ways that help LLMs produce more accurate, helpful, and on-target responses.

In item development, prompt engineering ensures that generated items are educationally sound, technically accurate, appropriately challenging, and consistently aligned with learning objectives.

#### Prompting Strategies
Effective prompt engineering for item generation employs various strategic approaches, each designed to optimize specific aspects of LLM output.

The working loop looks like this:

```
write a prompt  →  read the output  →  spot what is wrong  →  reword  →  read again  →  …
```

Three strategies do most of the work, and all three predate the current generation of models:

| Strategy | Prompt details |
|---|---|
| **Zero-shot** | Instruction without example. |
| **Few-shot** | Instruction with two to five examples of the desired output. |
| **Chain-of-thought** | Instruction with a request to reason before answering. |


##### **Zero-shot Prompting**
This is the simplest prompting strategy where we give the AI instructions
without any examples. It's like asking someone to do a task by just
explaining what you want, without showing them an example of expected output.

**When to use Zero-Shot:**
1. Quick prototyping and testing
2. Simple, straightforward tasks
3. When you don't have good examples ready
4. For creative tasks where you want variety

**Limitations:**
1. Less consistent formatting
2. May misunderstand complex requirements
3. Output quality can vary

##### **Few-shot Prompting**
Few-shot prompting is like teaching by showing examples. Instead of just
telling the AI what to do, we show it a few examples of inputs and their
desired outputs, and it learns the pattern.

**When to use Few-Shot:**
1. When you need consistent formatting
2. When you have good examples available
3. For complex tasks with specific patterns
4. When zero-shot isn't giving consistent results

**Benefits:**
1. More consistent output format
2. Better understanding of requirements
3. Can encode domain expertise through examples
4. Reduces ambiguity

**Limitations:**
1. Requires good quality examples
2. Uses more tokens (costs more)
3. Can overfit to example patterns

##### **Chain-of-Thought Prompting**
Chain-of-thought prompting guides the AI through explicit step-by-step reasoning processes, making the model's thinking visible and systematic. This approach is particularly valuable for complex assessment item development where multiple considerations must be balanced.

**When to use Chain-of-Thought:**
1. For complex assessment scenarios requiring multi-step analysis
2. When generating items that test higher-order thinking skills
3. For quality assurance and validation of item construction
4. When transparency in the generation process is important
5. For developing items with sophisticated distractors

**Benefits:**
1. Improved reasoning quality: Forces systematic consideration of all aspects
2. Transparency: Makes the generation process auditable and reviewable
3. Better distractor development: Explicit focus on misconception-based options
4. Quality control: Each step can be evaluated independently

**Limitations:**
1. Longer prompts and responses cost more token usage
2. May generate unnecessarily complex items
3. More processing time required
4. Requires careful prompt structure to maintain consistency










The literature says the choice matters. Chan et al. (2025) compared standard prompting against
chain-of-thought with worked examples and chain-of-thought expressed as code, on GPT-3.5 and GPT-4,
and found the chain-of-thought variants produced better items — with the largest gains on getting
the *key* right, which is the failure that matters most for a scored item.

So this is a real technique with measurable effects. We are going to use it properly, and then look
carefully at what it still does not give us.

In [1]:
# Install langchain and langchain google module
%pip install -q -U langchain langchain-google-genai langchain_groq

print("Install finished.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.60.0 which is incompatible.
Install finished.


In [2]:
import os
import re

from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

#GEMINI
GEMINI_MODEL = "gemini-3.5-flash"
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# temperature 0.7 is deliberate: it is what a person exploring in a chat window gets.
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.7)

#groq
GROQ_MODEL = "allam-2-7b"
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# temperature 0.7 is deliberate: it is what a person exploring in a chat window gets.
llm = ChatGroq(model=GROQ_MODEL, temperature=0.7)

def diagnose(e):
    """Turn an exception into the most likely cause, in plain language."""
    msg = str(e).lower()
    if "429" in msg or "quota" in msg or "rate limit" in msg:
        return "Free-tier quota or rate limit hit. Wait 60 seconds and re-run this cell."
    if "404" in msg or "not found" in msg:
        return f"Model '{GEMINI_MODEL}' not found. Run the model-list cell in notebook 0.0."
    if "401" in msg or "403" in msg or "api key" in msg:
        return "Key problem. Check GOOGLE_API_KEY in Colab Secrets, with notebook access on."
    return "Unexpected error — read the detail line below."


def ask(prompt, model=llm):
    """Send a plain-text prompt, return plain text. Never raises."""
    try:
        return model.invoke(prompt).content
    except Exception as e:
        return f"[call failed] {diagnose(e)}\n  Details: {type(e).__name__}: {e}"


print(f"Ready. Model: {GEMINI_MODEL}")
print(f"Ready. Model: {GROQ_MODEL}")

Ready. Model: gemini-3.5-flash
Ready. Model: allam-2-7b


## 1️.0 Zero-shot


In [8]:
ZERO_SHOT = (
    """Write a multiple-choice question for 8th grade science about photosynthesis.
    Give four options and indicate the correct answer."""
)

print(ask(ZERO_SHOT))

Which of the following best describes the main purpose of photosynthesis in plants?

A) Converting oxygen into carbon dioxide
B) Converting sunlight into chemical energy
C) Converting water into carbohydrates
D) Converting minerals into organic matter

Correct answer: B) Converting sunlight into chemical energy 


Read it as an item writer, not as a reader. Typical things to notice:

- The **format** is whatever the model felt like — `A)` or `A.` or a bulleted list; the key at the
  bottom, or in bold, or in a sentence.
- The **distractors** are often wrong in ways no eighth grader is wrong.
- Nothing states **which standard** the item measures, or at what cognitive level.

## 1️.1 Zero-shot (Detailed Instruction)


In [10]:
ZERO_SHOT_long = (
    """You are an experienced item writer creating assessment items for middle school science.
    Write one multiple-choice question for 8th grade students on the topic of photosynthesis.
    The item should align with NGSS MS-LS1-6 (constructing an explanation for the role of
    photosynthesis in the cycling of matter and flow of energy into and out of organisms).

    Requirements for the stem:
    - Write a clear, complete question that can be answered without looking at the options.

    Requirements for the options:
    - Provide exactly four options labeled A, B, C, and D.
    - Include exactly one correct answer that is scientifically accurate and unambiguous.
    - Avoid clues that point to the key, such as repeated words from the stem.

    Output format:
    "Question: <stem>\n"
    "A. <option>\n"
    "B. <option>\n"
    "C. <option>\n"
    "D. <option>\n"
    "Correct Answer: <letter>\n"""
)

print(ask(ZERO_SHOT_long))

Question: Photosynthesis is a process that occurs in plants, algae, and some bacteria, converting light energy into chemical energy stored in the form of glucose. This process plays a significant role in the cycling of matter and flow of energy into and out of organisms. According to the scientific explanation, what is the main product of photosynthesis?

A. Oxygen
B. Carbon dioxide
C. Water
D. Methane

Correct Answer: A 


## 2️. Few-shot

Show the model what you want instead of describing it. Two well-chosen examples.

In [11]:
FEW_SHOT = """You are an experienced 8th grade science item writer.
Write one multiple-choice item in exactly the style of the examples below.

EXAMPLE 1
Stem: A plant is kept in a dark closet for two weeks. Which change is most likely?

A. The plant gains mass because it stores water.
B. The plant loses mass because it cannot carry out photosynthesis.
C. The plant grows faster because it is not stressed by light.
D. The plant stays the same because roots supply all its food.
Answer: B
Rationale: Without light the plant cannot photosynthesize and consumes stored sugars.

EXAMPLE 2
Stem: Which substance is released into the air during photosynthesis?

A. Nitrogen
B. Carbon dioxide
C. Oxygen
D. Water vapor
Answer: C
Rationale: Oxygen is produced when water molecules are split during the light reactions.

NOW WRITE ONE NEW ITEM
Topic: the role of chlorophyll in capturing light energy
"""

print(ask(FEW_SHOT))

Stem: Chlorophyll plays a crucial role in the process of photosynthesis. What function does it primarily serve?

A. Converting water into oxygen
B. Absorbing light energy for chemical reactions
C. Regulating the growth and development of plants
D. Producing glucose for plant consumption

Answer: B
Rationale: Chlorophyll is a pigment found in plant cells that absorbs light energy from the sun, facilitating the conversion of carbon dioxide and water into glucose and oxygen during photosynthesis. 


There is usually a visible improvement — the format tracks the examples, and the distractors are
closer to real student thinking. This is the single cheapest upgrade available to most programs,
and it is the reason "just add two examples" is the most useful sentence in prompt engineering.

## 3️. Chain-of-thought

Ask for reasoning before the answer. For item writing, the useful version is not "think step by
step" in the abstract — it is asking for the intermediate object you actually want: the student
misconceptions that the distractors should represent.

In [12]:
CHAIN_OF_THOUGHT = """You are an experienced 8th grade science item writer.

Work in two steps.

Step 1. List three specific misconceptions 8th grade students hold about photosynthesis.
Number them.

Step 2. Write one multiple-choice item on photosynthesis in which each of the three
distractors corresponds to one of the misconceptions from Step 1. Label the stem, the four
options A-D, the answer, and a one-line rationale for each option.
"""

print(ask(CHAIN_OF_THOUGHT))

Step 1: Three specific misconceptions 8th grade students may hold about photosynthesis are:

1. Misconception: Photosynthesis is solely about producing oxygen.
2. Misconception: All green plants perform photosynthesis indoors only.
3. Misconception: Light is not necessary for photosynthesis to occur.

Step 2: Multiple-choice item on photosynthesis with distractors addressing the misconceptions:

Stem: Photosynthesis is a process where plants convert sunlight, carbon dioxide, and water into which of the following?

A. Oxygen and glucose (sugar)
B. Water and carbon dioxide
C. Heat and light
D. Fatty acids and proteins

Answer: A. Oxygen and glucose (sugar)

Rationale: The correct answer is A because, during photosynthesis, plants convert sunlight, carbon dioxide, and water into oxygen and glucose (a type of sugar). 

Distractors:

B. Water and carbon dioxide (This distractor addresses the misconception that plants only produce oxygen and not glucose)
C. Heat and light (This distractor ad

The distractors now have a *reason to exist*. Each one is a wrong answer some
real student would actually choose, and the rationale tells a reviewer why. That is the difference
between four options and one key plus three pieces of filler.

##

## 4️. Hands-on Activity

In [34]:
LOCALITY = "the United States"
ACTIVITY_PROMPT = (
    f"""You are an experienced item writer for 12th grade civics assessments in {LOCALITY}.

    Requirements for the stem:
    - Write a clear, complete question that can be answered without looking at the options.
    - The stem should be no more than two sentences.

    Requirements for the options:
    - Provide exactly four options labeled A, B, C, and D.

    Output format:
    "Question: <stem>\n"
    "A. <option>\n"
    "B. <option>\n"
    "C. <option>\n"
    "D. <option>\n"
    "Correct Answer: <letter>\n

    Work in two steps.

    Step 1. List three specific misconceptions 12th grade students in {LOCALITY} hold about systems of government.
    Number them.

    Step 2. Write one multiple-choice item on systems of government in which each of the three
    distractors corresponds to one of the misconceptions from Step 1. Label the stem, the four
    options A-D, the answer, and a one-line rationale for each option. There must be only one
    correct answer."""

)

print(ask(ACTIVITY_PROMPT))

Step 1: List three specific misconceptions 12th grade students in the United States hold about systems of government:

1. Misconception 1: All governments in the US are based on the same principles and structure, regardless of location.
2. Misconception 2: The US government is a monarchy, with a king or queen as the head of state.
3. Misconception 3: The US has only one form of government, which is a direct democracy.

Step 2: Write a multiple-choice item addressing these misconceptions:

Question: Which of the following statements best describes the US government system?

A. The US government is a pure democracy where citizens directly make decisions.
B. The US government has a monarchy with a king or queen as the head of state.
C. All governments in the US are based on the same principles and structure regardless of location.
D. The US government is a federal republic, with a president as the head of state.

Correct Answer: D (Federal Republic)

Rationale for Option A: This option is

## 5. Generating multiple items

Everything above was a single run. Operationally you will not generate one item; you will generate
hundreds, and the question is whether the thing you tuned holds up across a batch.

#### One item JSON

In [35]:
## genrating with json structure
CHAIN_OF_THOUGHT_multiple = """You are an experienced 8th grade science item writer.

Work in two steps.

Step 1. List three specific misconceptions 8th grade students hold about photosynthesis.
Number them.

Step 2. Write one multiple-choice item on photosynthesis in which each of the three
distractors corresponds to one of the misconceptions from Step 1. Label the stem, the four
options A-D, the answer, and a one-line rationale for each option.

Format your response as a JSON object with this structure:
{{
  "stem": "[question stem]",
  "option_a": "[option]",
  "option_b": "[option]",
  "option_c": "[option]",
  "option_d": "[option]",
  "correct_answer": "[single letter: A, B, C, or D]",
  "rationale_a": "[why option A is correct or incorrect, and which misconception it targets]",
  "rationale_b": "[why option B is correct or incorrect, and which misconception it targets]",
  "rationale_c": "[why option C is correct or incorrect, and which misconception it targets]",
  "rationale_d": "[why option D is correct or incorrect, and which misconception it targets]",
  "misconception1": "[misconception]",
  "misconception2": "[misconception]",
  "misconception3": "[misconception]"
}}

Respond with the JSON object only. Leave out any extra text, explanations, or markdown such as code fences.
"""

print(ask(CHAIN_OF_THOUGHT_multiple))

{
  "stem": "A student believes that plants can only convert sunlight into energy during the daytime. They ask, \"Can plants convert sunlight into energy at night?\"",
  "option_a": "A. Yes, plants can convert sunlight into energy day and night.",
  "option_b": "B. Yes, plants can convert sunlight into energy during daytime only.",
  "option_c": "C. No, plants can only convert sunlight into energy during the daytime.",
  "option_d": "D. No, plants cannot convert sunlight into energy at all.",
  "correct_answer": "C",
  "rationale_a": "Option A is incorrect because plants only convert sunlight into energy during the daytime, not at night when photosynthesis requires light for the process.",
  "rationale_b": "Option B is incorrect because photosynthesis requires light from sunlight for energy production, so the statement is false.",
  "rationale_c": "Option C is correct because it addresses the misconception that plants can only convert sunlight into energy during the daytime.",
  "ratio

#### Five item JSON

In [36]:
import json
import time
from datetime import datetime

def parse_json(text):
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

rows = []
n_items = 5
max_tries = 3

for i in range(1, n_items + 1):
    item = None
    for attempt in range(1, max_tries + 1):
        try:
            output = ask(CHAIN_OF_THOUGHT_multiple)
            item = parse_json(output)
            break
        except Exception as e:
            print(f"Item {i}, attempt {attempt}: {type(e).__name__}, retrying...")
            time.sleep(3)

    if item is None:
        print(f"Item {i} failed after {max_tries} tries.")
        continue

    item["correct_answer"] = str(item.get("correct_answer", "")).strip()[:1].upper()
    rows.append({"id": i, **item})
    print(f"Item {i} done")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"cot_items_{timestamp}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(rows)} of {n_items} items to {filename}")

Item 1 done
Item 2 done
Item 3 done
Item 4 done
Item 5 done

Saved 5 of 5 items to cot_items_20261005_183405.json


#### Read generated JSON output

In [37]:
import pandas as pd

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

,id,stem,option_a,option_b,option_c,option_d,correct_answer,rationale_a,rationale_b,rationale_c,rationale_d,misconception1,misconception2,misconception3
0,1,Which of the following statements about photos...,Photosynthesis only occurs in plants.,Light energy is not needed for photosynthesis.,Photosynthesis is the process by which plants ...,Carbon dioxide is the only gas increased durin...,B,Option A is incorrect because photosynthesis a...,Option B is incorrect because light energy is ...,Option C is incorrect because plants do not pr...,Option D is incorrect because water is also a ...,Photosynthesis occurs only in plants.,Light energy is not needed for photosynthesis.,Photosynthesis produces oxygen as its main out...
1,2,Which of the following statements about photos...,Photosynthesis occurs only in dark-colored lea...,Photosynthesis is the process of converting su...,Photosynthesis requires a lot of water and sun...,Photosynthesis involves the production of toxi...,C,The correct answer is option C because it accu...,Option B is incorrect because photosynthesis i...,Option C is the correct answer as it mentions ...,Option D is incorrect because photosynthesis a...,Photosynthesis occurs only in dark-colored lea...,Photosynthesis is the process of converting su...,Photosynthesis involves the production of toxi...
2,3,"In photosynthesis, which of the following stat...","When plants take in carbon dioxide, they relea...",Photosynthesis only occurs in plants that are ...,Photosynthesis happens in the roots of plants.,Plants produce glucose during photosynthesis.,D,Students may believe that plants release oxyge...,The misconception here is that photosynthesis ...,The misconception involving the roots is that ...,The correct statement is that plants produce g...,Plants release oxygen when they absorb carbon ...,Photosynthesis only occurs in direct sunlight,Photosynthesis takes place in the roots of plants
3,4,Which of the following statements best describ...,3,1,2,4,2,Option 1 suggests that plants release carbon d...,Option 2 correctly describes the main process ...,Option 3 is incorrect because it mentions the ...,Option 4 is incorrect as it mentions the relea...,Plants release oxygen and hydrogen.,"Photosynthesis involves sunlight, water, and n...",Photosynthesis produces sulfur.
4,5,Which of the following statements about photos...,1. Plants can only convert light into chemical...,2. Oxygen is not released during photosynthesi...,3. Carbon dioxide is not absorbed by plants. T...,Plants can only convert light into chemical en...,B,Option A addresses misconception 1 that plants...,"Option B addresses misconception 2, which clai...","Option C addresses misconception 3, stating th...",NaN,Plants only convert light into chemical energy.,Oxygen is not released during photosynthesis.,Plants do not absorb carbon dioxide
